## Main design

In [1]:
import qiskit_metal
qiskit_metal.about()


Qiskit Metal        0.8.1

Basic
____________________________________
 Python              3.11.15 (main, May  4 2026, 21:17:00) [MSC v.1944 64 bit (AMD64)]
 Platform            Windows AMD64
 Installation path   C:\Users\DELL\chip-design\metal-flow\quantum-metal\src\qiskit_metal

Packages
____________________________________
 Numpy               1.26.4
 Qutip               5.3.0

Rendering
____________________________________
 Matplotlib          3.11.0

GUI
____________________________________
 PySide6 version     6.11.1
 Qt version          6.11.1
 SIP version         Not installed

IBM Quantum Team


'\nQiskit Metal        0.8.1\n\nBasic\n____________________________________\n Python              3.11.15 (main, May  4 2026, 21:17:00) [MSC v.1944 64 bit (AMD64)]\n Platform            Windows AMD64\n Installation path   C:\\Users\\DELL\\chip-design\\metal-flow\\quantum-metal\\src\\qiskit_metal\n\nPackages\n____________________________________\n Numpy               1.26.4\n Qutip               5.3.0\n\nRendering\n____________________________________\n Matplotlib          3.11.0\n\nGUI\n____________________________________\n PySide6 version     6.11.1\n Qt version          6.11.1\n SIP version         Not installed\n\nIBM Quantum Team'

In [2]:
# Cell 1 — Imports + design skeleton
from qiskit_metal import designs, draw, MetalGUI, Dict
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder

design = designs.DesignPlanar()
design.overwrite_enabled = True

design.chips.main.size['size_x'] = '11mm'
design.chips.main.size['size_y'] = '11mm'

design.chips.main.size['size_z'] = '-280um'
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '11mm',
  'size_y': '11mm',
  'size_z': '-280um',
  'sample_holder_top': '890um',
  'sample_holder_bottom': '1650um'}}

In [3]:
# Cell 2 (updated) — Q0–Q3 unchanged (TransmonPocket, 3 pads each fits the 4-corner limit).
# Q4 rebuilt as TransmonPocket6 with 5 angularly-placed connection pads.

from qiskit_metal.qlibrary.qubits.transmon_pocket_6 import TransmonPocket6


arm = "2 mm"


q0 = TransmonPocket(design, 'Q0', options=Dict(
    pos_x=f'-{arm}', pos_y=f'{arm}', orientation=90, pad_width='450 um', pad_height= "100 um", pad_gap = '20 um',
    connection_pads=Dict(
        to_Q4  =Dict(loc_W=-1, loc_H=-1),
        to_Q2  =Dict(loc_W=-1, loc_H=+1),
        readout=Dict(loc_W=+1, loc_H=+1),
    )))


q1 = TransmonPocket6(design, 'Q1', options=Dict(
    pos_x=f'{arm}', pos_y=f'{arm}',orientation=-90, pad_width='450 um', pad_height= "100 um", pad_gap = '20 um',
    connection_pads=Dict(
        readout=Dict(loc_W=-1, loc_H=+1),
        to_Q3  =Dict(loc_W=+1, loc_H=+1),
        to_Q4  =Dict(loc_W=+1, loc_H=-1),
    )))

q2 = TransmonPocket6(design, 'Q2', options=Dict(
    pos_x=f'-{arm}', pos_y=f'-{arm}',orientation=-90, pad_width='450 um', pad_height= "100 um", pad_gap = '20 um',
    connection_pads=Dict(
        readout=Dict(loc_W=+1, loc_H=-1),
        to_Q0  =Dict(loc_W=-1, loc_H=-1),
        to_Q4  =Dict(loc_W=-1, loc_H=+1),
    )))

q3 = TransmonPocket6(design, 'Q3', options=Dict(
    pos_x=f'{arm}', pos_y=f'-{arm}',orientation=90, pad_width='450 um', pad_height= "100 um", pad_gap = '20 um',
    connection_pads=Dict(
        readout=Dict(loc_W=-1, loc_H=-1),
        to_Q1  =Dict(loc_W=+1, loc_H=-1),
        to_Q4  =Dict(loc_W=+1, loc_H=+1),
    )))

# TransmonPocket6: connection_pads placed by angle (connector_location, degrees, CCW from +x).
# connector_type='0' = claw connector (matches TransmonPocket's pin style for routing).
q4 = TransmonPocket6(design, 'Q4', options=Dict(
    pos_x='0mm', pos_y='0mm', orientation=0, pad_width='450 um', pad_height= "100 um", pad_gap = '20 um',
    connection_pads=Dict(
        to_Q0  = Dict(loc_W=-1, loc_H=+1),
        to_Q1  = Dict(loc_W=+1, loc_H=+1),
        to_Q2  = Dict(loc_W=-1, loc_H=-1),
        to_Q3  = Dict(loc_W=+1, loc_H=-1),
        readout= Dict(loc_W=+0, loc_H=+1),
    )))

In [4]:
# Cell 4 (updated) — direct qubit-qubit couplers, now meandered instead of straight.
to_q4_meander_common = Dict(trace_width='10um', trace_gap='6um', fillet='69um',
                       meander=Dict(spacing='200um'),
                       lead=Dict(start_straight='600um', end_straight='0um'))




c02_jogsS = Dict()
c02_jogsS[0] = ["R", "300um"]

c02_jogsE = Dict()
c02_jogsE[0] = ["L", "300um"]

c02_meander_opts = Dict(
    advanced= Dict(avoid_collision= "true"),
    meander=Dict(spacing= "300um", asymmetry="-100um"),
    snap="true",
    lead= Dict(
        start_straight= "0.2mm",
        end_straight= "0.2mm",
        start_jogged_extension= c02_jogsS,
        end_jogged_extension= c02_jogsE,
    ),
    fillet='99 um',
)


c02 = RouteMeander(design, 'C02', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q0', pin='to_Q2'),
                     end_pin=Dict(component='Q2', pin='to_Q0')),
    total_length='10mm', **c02_meander_opts))




c13_jogsS = Dict()
c13_jogsS[0] = ["L", "300um"]

c13_jogsE = Dict()
c13_jogsE[0] = ["R", "300um"]

c13_meander_opts = Dict(
    advanced= Dict(avoid_collision= "true"),
    meander=Dict(spacing= "300um", asymmetry="100um"),
    snap="true",
    lead= Dict(
        start_straight= "0.2mm",
        end_straight= "0.2mm",
        start_jogged_extension= c13_jogsS,
        end_jogged_extension= c13_jogsE,
    ),
    fillet='99 um',
)


c13 = RouteMeander(design, 'C13', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q1', pin='to_Q3'),
                     end_pin=Dict(component='Q3', pin='to_Q1')),
    total_length='10mm', **c13_meander_opts))

c04 = RouteMeander(design, 'C04', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q0', pin='to_Q4'),
                     end_pin=Dict(component='Q4', pin='to_Q0')),
    total_length='10mm', asymmetry='0um', **to_q4_meander_common))

c41 = RouteMeander(design, 'C41', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q1', pin='to_Q4'),
                     end_pin=Dict(component='Q4', pin='to_Q1')),
    total_length='10mm', asymmetry='0um', **to_q4_meander_common))

c24 = RouteMeander(design, 'C24', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q2', pin='to_Q4'),
                     end_pin=Dict(component='Q4', pin='to_Q2')),
    total_length='10mm', asymmetry='0um', **to_q4_meander_common))

c43 = RouteMeander(design, 'C43', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q3', pin='to_Q4'),
                     end_pin=Dict(component='Q4', pin='to_Q3')),
    total_length='10mm', asymmetry='0um', **to_q4_meander_common))

# If you get a "total_length too short for meander" error, bump total_length up —
# it must exceed the straight-line pin-to-pin distance plus the lead lengths.

In [5]:
# Cell 5 (updated) — Launchpads (corrected orientation) + feedline built from
# LaunchpadWirebond -> CoupledLineTee -> CoupledLineTee -> LaunchpadWirebond.
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

lp_opts = Dict(trace_width='10um', trace_gap='6um', lead_length='30um')

# Top pads face south (270deg) into the chip; bottom pads face north (90deg) into the chip.
tl0_in  = LaunchpadWirebond(design, 'TL_0_IN',  options=Dict(pos_x='-5mm', pos_y='5mm',  orientation='270', **lp_opts))
tl0_out = LaunchpadWirebond(design, 'TL_0_OUT', options=Dict(pos_x='-5mm', pos_y='-5mm', orientation='90',  **lp_opts))
tl1_in  = LaunchpadWirebond(design, 'TL_1_IN',  options=Dict(pos_x='5mm',  pos_y='5mm',  orientation='270', **lp_opts))
tl1_out = LaunchpadWirebond(design, 'TL_1_OUT', options=Dict(pos_x='5mm',  pos_y='-5mm', orientation='90',  **lp_opts))

tee_opts = Dict(prime_width='10um', prime_gap='6um',
                 second_width='10um', second_gap='6um',
                 coupling_space='4um', coupling_length='200um', fillet='49um')

# Left side: prime line runs vertically -> rotate tee 90deg so second_end points east (toward Q0/Q2, at x=-3).
clt_q0 = CoupledLineTee(design, 'CLT_Q0', options=Dict(pos_x='-5mm', pos_y='3mm',  orientation='90',  mirror=False, **tee_opts))
clt_q2 = CoupledLineTee(design, 'CLT_Q2', options=Dict(pos_x='-5mm', pos_y='-3mm', orientation='90',  mirror=False, **tee_opts))

# Right side: rotate tee -90deg so second_end points west (toward Q1/Q3, at x=3).
clt_q1 = CoupledLineTee(design, 'CLT_Q1', options=Dict(pos_x='5mm', pos_y='3mm',  orientation='270', mirror=False, **tee_opts))
clt_q3 = CoupledLineTee(design, 'CLT_Q3', options=Dict(pos_x='5mm', pos_y='-3mm', orientation='270', mirror=False, **tee_opts))

# After gui.rebuild(), check that each tee's coupled stub (second_end) points at its
# qubit, not away from it — if it's mirrored, flip mirror=True and re-rebuild.

feed_opts = Dict(trace_width='10um', trace_gap='6um', fillet='0um')

tl0_seg1 = RouteStraight(design, 'TL_0_seg1', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='TL_0_IN', pin='tie'),
                     end_pin=Dict(component='CLT_Q0', pin='prime_end')), **feed_opts))
tl0_seg2 = RouteStraight(design, 'TL_0_seg2', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='CLT_Q0', pin='prime_start'),
                     end_pin=Dict(component='CLT_Q2', pin='prime_end')), **feed_opts))
tl0_seg3 = RouteStraight(design, 'TL_0_seg3', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='CLT_Q2', pin='prime_start'),
                     end_pin=Dict(component='TL_0_OUT', pin='tie')), **feed_opts))

tl1_seg1 = RouteStraight(design, 'TL_1_seg1', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='TL_1_IN', pin='tie'),
                     end_pin=Dict(component='CLT_Q1', pin='prime_start')), **feed_opts))
tl1_seg2 = RouteStraight(design, 'TL_1_seg2', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='CLT_Q1', pin='prime_end'),
                     end_pin=Dict(component='CLT_Q3', pin='prime_start')), **feed_opts))
tl1_seg3 = RouteStraight(design, 'TL_1_seg3', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='CLT_Q3', pin='prime_end'),
                     end_pin=Dict(component='TL_1_OUT', pin='tie')), **feed_opts))

In [6]:
# Cell 6 (updated) — Resonators R0-R3: CoupledLineTee.second_end -> qubit.readout (meandered).
# R4: Q4's own 5th pad, left as an open (unconnected) resonator per our earlier assumption
# that the diagram doesn't show a feedline for Q4.

res_opts = Dict(trace_width='10um', trace_gap='6um', fillet='99um',
                 lead=Dict(start_straight='60um', end_straight='60um'))

r0 = RouteMeander(design, 'R0', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q0', pin='readout'),
                     end_pin=Dict(component='CLT_Q0', pin='second_end')),
    total_length='22mm', meander=Dict(spacing='200um', asymmetry='-700um'), **res_opts))

r1 = RouteMeander(design, 'R1', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q1', pin='readout'),
                     end_pin=Dict(component='CLT_Q1', pin='second_end')),
    total_length='22mm', meander=Dict(spacing='200um', asymmetry='700um'), **res_opts))

r2 = RouteMeander(design, 'R2', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q2', pin='readout'),
                     end_pin=Dict(component='CLT_Q2', pin='second_end')),
    total_length='22mm', meander=Dict(spacing='200um', asymmetry='700um'), **res_opts))

r3 = RouteMeander(design, 'R3', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q3', pin='readout'),
                     end_pin=Dict(component='CLT_Q3', pin='second_end')),
    total_length='22mm', meander=Dict(spacing='200um', asymmetry='-700um'), **res_opts))


r4_lp_opts = Dict(trace_width='10um', trace_gap='6um', lead_length='30um')

r4_lp = LaunchpadWirebond(design, 'r4_lp', options=Dict(pos_x='0mm',  pos_y='5mm', orientation='-90',  **r4_lp_opts))


r4 = RouteMeander(design, 'R4', options=Dict(
    pin_inputs=Dict(start_pin=Dict(component='Q4', pin='readout'),
                     end_pin=Dict(component='r4_lp', pin='tie')),
    total_length='22mm', trace_width='10um', trace_gap='6um', fillet='99um',
                 lead=Dict(start_straight='1500um', end_straight='0um')))

In [7]:
design.rebuild()

In [8]:
from qiskit_metal import gui

gui(design)
